# LoRA fine-tune: Hinglish expense classifier

Qwen2.5-1.5B-Instruct + LoRA on kharcha's 15 categories. Runs on a free Colab T4 in ~20-30 min.

Before running: Runtime -> Change runtime type -> T4 GPU. Upload `data/train.jsonl` and `data/val.jsonl` when prompted (or mount Drive).

In [ ]:
%pip install -q unsloth trl peft datasets
# If unsloth's API has drifted, see https://github.com/unslothai/unsloth for the current quickstart.

In [ ]:
from google.colab import files
print("Upload train.jsonl and val.jsonl")
files.upload()

In [ ]:
import json
from datasets import Dataset

CATEGORIES = [
    "Food & Dining", "Groceries", "Transport", "Shopping",
    "Utilities & Bills", "Subscriptions", "Health", "Entertainment",
    "Rent & Home", "Education", "Travel", "Personal Care",
    "Gifts & Family", "Cash Withdrawal", "Other",
]
INSTRUCTION = (
    "Classify this expense entry into exactly one category from this list: "
    + "; ".join(CATEGORIES)
    + ". Respond with only the category name, nothing else."
)

def load_rows(path):
    return [json.loads(x) for x in open(path)]

train_rows, val_rows = load_rows("train.jsonl"), load_rows("val.jsonl")
print(len(train_rows), "train /", len(val_rows), "val")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
def to_text(row):
    messages = [
        {"role": "user", "content": f'{INSTRUCTION}\n\nEntry: "{row["text"]}"'},
        {"role": "assistant", "content": row["label"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in train_rows])

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    dataset_text_field="text", max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=16, gradient_accumulation_steps=2,
        num_train_epochs=2, learning_rate=2e-4, warmup_ratio=0.05,
        logging_steps=20, output_dir="out", optim="adamw_8bit",
        lr_scheduler_type="cosine", seed=13, fp16=True, report_to="none",
    ),
)
trainer.train()

In [ ]:
# Quick val accuracy before pushing
import torch
FastLanguageModel.for_inference(model)

def predict(text):
    messages = [{"role": "user", "content": f'{INSTRUCTION}\n\nEntry: "{text}"'}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

sample = val_rows[:150]
correct = sum(predict(r["text"]).lower().startswith(r["label"].lower()[:6]) for r in sample)
print(f"val accuracy (n={len(sample)}): {correct/len(sample):.3f}")

In [ ]:
# Push merged model + GGUF to your Hugging Face account
from huggingface_hub import login
login()  # paste a WRITE token from hf.co/settings/tokens

REPO = "YOUR_HF_USERNAME/hinglish-expense-classifier-1.5b"
model.push_to_hub_merged(REPO, tokenizer, save_method="merged_16bit")
model.push_to_hub_gguf(REPO + "-gguf", tokenizer, quantization_method="q4_k_m")
print("pushed:", REPO)

Back on your Mac:

```bash
python evaluate.py --backend hf --model YOUR_HF_USERNAME/hinglish-expense-classifier-1.5b
```